# Optimization Modelling — Task 2

Realistic capacity expansion with piecewise marginal costs and minimum distance constraints between facilities.


## 1. Imports

In [ ]:
# ── Run this cell if running LOCALLY ──────────────────────────────────────────
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB

In [2]:
# ── Run this cell if running on GOOGLE COLAB ──────────────────────────────────
import os
import pandas as pd
import numpy as np
import gurobipy as gp
from gurobipy import GRB
from google.colab import files

# Upload data files
uploaded = files.upload()  # select all 5 CSV files

# Remove any old Gurobi license files
for f in os.listdir('/content'):
    if 'gurobi' in f.lower() and '.lic' in f.lower():
        os.remove(f'/content/{f}')
        print(f"Removed: {f}")

# Upload your gurobi.lic
lic_upload = files.upload()  # select the gurobi.lic you downloaded
os.environ['GRB_LICENSE_FILE'] = '/content/gurobi.lic'

# Test the license
m_test = gp.Model()
m_test.addVars(10000, vtype='B')
m_test.update()
print("License OK — variables:", m_test.NumVars)
m_test.dispose()

ModuleNotFoundError: No module named 'gurobipy'

## 2. Load Data

- **Task 1** — population and income/employment (already cleaned and ZIP-aligned)
- **Task 2** — ZIP-aligned child care facilities and potential new locations (with coordinates)


In [ ]:
# ── Run this cell if running LOCALLY ──────────────────────────────────────────
# ── From Task 1 EDA ───────────────────────────────────────────────────────────────
population_nyc      = pd.read_csv('../data/task_1/population_nyc.csv')
avg_income_nyc      = pd.read_csv('../data/task_1/avg_income_nyc.csv')
employment_rate_nyc = pd.read_csv('../data/task_1/employment_rate_nyc.csv')

# ── From Task 2 EDA ───────────────────────────────────────────────────────────────
child_care_regulated_nyc = pd.read_csv('../data/task_2/child_care_regulated_nyc.csv')
potential_locations_nyc  = pd.read_csv('../data/task_2/potential_locations_nyc.csv')

In [ ]:
# ── Run this cell if running on GOOGLE COLAB ──────────────────────────────────
population_nyc           = pd.read_csv('/content/population_nyc.csv')
avg_income_nyc           = pd.read_csv('/content/avg_income_nyc.csv')
employment_rate_nyc      = pd.read_csv('/content/employment_rate_nyc.csv')
child_care_regulated_nyc = pd.read_csv('/content/child_care_regulated_nyc.csv')
potential_locations_nyc  = pd.read_csv('/content/potential_locations_nyc.csv')

In [ ]:
pop = population_nyc.rename(columns={'-5': 'pop_0_5', '6-12': 'pop_6_12'})[['zipcode', 'pop_0_5', 'pop_6_12']]
pop['pop_2wk_12yr'] = pop['pop_0_5'] + pop['pop_6_12']
pop.head()

,zipcode,pop_0_5,pop_6_12,pop_2wk_12yr
0,10001,744,1255,1999
1,10002,2142,4645,6787
2,10003,1440,1510,2950
3,10004,433,262,695
4,10005,484,318,802


In [ ]:
child_care_regulated_nyc['slots_0_5'] = (
    child_care_regulated_nyc['infant_capacity'] +
    child_care_regulated_nyc['toddler_capacity'] +
    child_care_regulated_nyc['preschool_capacity']
)

slots = child_care_regulated_nyc.groupby('zipcode').agg(
    total_slots=('total_capacity', 'sum'),
    slots_0_5=('slots_0_5', 'sum')
).reset_index()
slots.head()

,zipcode,total_slots,slots_0_5
0,10001,609,0
1,10002,4729,18
2,10003,1995,0
3,10004,263,0
4,10005,39,0


In [ ]:
df = pop.merge(slots, on='zipcode', how='left')
df = df.merge(avg_income_nyc.rename(columns={'average income': 'avg_income'}), on='zipcode', how='left')
df = df.merge(employment_rate_nyc.rename(columns={'employment rate': 'emp_rate'}), on='zipcode', how='left')
df[['total_slots', 'slots_0_5']] = df[['total_slots', 'slots_0_5']].fillna(0)
df.head()

,zipcode,pop_0_5,pop_6_12,pop_2wk_12yr,total_slots,slots_0_5,avg_income,emp_rate
0,10001,744,1255,1999,609.0,0.0,102878.033603,0.595097
1,10002,2142,4645,6787,4729.0,18.0,59604.041165,0.520662
2,10003,1440,1510,2950,1995.0,0.0,114273.049645,0.497244
3,10004,433,262,695,263.0,0.0,132004.310345,0.506661
4,10005,484,318,802,39.0,0.0,121437.713311,0.665833


In [ ]:
df['demand'] = df.apply(
    lambda r: 'High' if (r['avg_income'] <= 60_000 or r['emp_rate'] >= 0.60) else 'Normal',
    axis=1
)
print(df['demand'].value_counts().to_string())
df[['zipcode', 'avg_income', 'emp_rate', 'demand']].head(10)

demand
High      97
Normal    83


,zipcode,avg_income,emp_rate,demand
0,10001,102878.033603,0.595097,Normal
1,10002,59604.041165,0.520662,High
2,10003,114273.049645,0.497244,Normal
3,10004,132004.310345,0.506661,Normal
4,10005,121437.713311,0.665833,High
5,10006,126377.118644,0.631692,High
6,10007,138853.904282,0.528910,Normal
7,10009,77133.233533,0.514567,Normal
8,10010,116272.698810,0.492749,Normal
9,10011,120420.792079,0.557000,Normal


In [ ]:
THRESHOLD_HIGH   = 1/2
THRESHOLD_NORMAL = 1/3
THRESHOLD_UNDER5 = 2/3

df['total_threshold'] = df.apply(
    lambda r: THRESHOLD_HIGH * r['pop_2wk_12yr'] if r['demand'] == 'High'
              else THRESHOLD_NORMAL * r['pop_2wk_12yr'],
    axis=1
)

df['fail_total']  = df['total_slots'] <= df['total_threshold']
df['fail_under5'] = df['slots_0_5']   <= THRESHOLD_UNDER5 * df['pop_0_5']
df['is_desert']   = (df['fail_total'] | df['fail_under5']) & (df['pop_2wk_12yr'] > 0)
print(f"Zips failing total capacity check : {df['fail_total'].sum()}")
print(f"Zips failing under-5 policy check : {df['fail_under5'].sum()}")
print(f"Total desert zip codes            : {df['is_desert'].sum()}")

Zips failing total capacity check : 162
Zips failing under-5 policy check : 180
Total desert zip codes            : 179


In [ ]:
df['total_slot_deficit'] = np.ceil(
    (df['total_threshold'] - df['total_slots']).clip(lower=0)
).astype(int)

df['under5_slot_deficit'] = np.ceil(
    (THRESHOLD_UNDER5 * df['pop_0_5'] - df['slots_0_5']).clip(lower=0)
).astype(int)
df[['zipcode', 'total_slots', 'total_threshold', 'total_slot_deficit',
    'slots_0_5', 'under5_slot_deficit']].head(10)

,zipcode,total_slots,total_threshold,total_slot_deficit,slots_0_5,under5_slot_deficit
0,10001,609.0,666.333333,58,0.0,496
1,10002,4729.0,3393.500000,0,18.0,1410
2,10003,1995.0,983.333333,0,0.0,960
3,10004,263.0,231.666667,0,0.0,289
4,10005,39.0,401.000000,362,0.0,323
5,10006,156.0,130.500000,0,14.0,72
6,10007,284.0,400.333333,117,0.0,404
7,10009,1784.0,1490.333333,0,18.0,1246
8,10010,234.0,1162.666667,929,0.0,948
9,10011,1956.0,1030.333333,0,42.0,764


## 3. Optimization Model

We formulate a **Mixed-Integer Linear Program (MILP)** to determine the least-cost strategy for eliminating child-care deserts across 179 NYC ZIP codes. Two levers are available:

1. **Expand existing facilities** — add slots to the 7,739 regulated facilities already operating in desert ZIPs.
2. **Build new facilities** — select among feasible potential locations, choosing a Small (100 slots), Medium (200 slots), or Large (400 slots) facility for each.

### 3.1 Distance Pre-processing

Before building the model, we pre-compute spatial feasibility using the **Haversine formula** to measure great-circle distances between locations. A minimum separation of **0.06 miles** is enforced between any two facilities (existing or new) in the same ZIP:

- **Infeasible locations**: any candidate site within 0.06 miles of an *existing* facility is ruled out entirely — it cannot be built.
- **Forbidden pairs**: any two *candidate* sites within 0.06 miles of each other form a forbidden pair — at most one of them can be selected. These are passed directly as constraints into the model.

In [ ]:
# ── Distance Preprocessing ─────────────────────────────────────────────────────
# Task 2 requires that no two facilities (existing or new) within the same ZIP
# are closer than 0.06 miles. We pre-compute this BEFORE building the Gurobi model
# so the model doesn't need to compute distances itself.
#
# This produces two outputs:
#   infeasible_locs : potential locations that are already too close to an
#                     existing facility → cannot be built at all
#   forbidden_pairs : pairs of potential locations too close to each other
#                     → at most one of them can be selected

MIN_DIST = 0.06    # minimum distance in miles
R_EARTH  = 3958.8  # Earth radius in miles

def haversine(lat1, lon1, lat2, lon2):
    """
    Vectorized Haversine distance matrix (miles).
    Returns D where D[i, j] = distance(point2[i], point1[j]).
    """
    lat1 = np.radians(np.asarray(lat1, dtype=float))
    lon1 = np.radians(np.asarray(lon1, dtype=float))
    lat2 = np.radians(np.asarray(lat2, dtype=float))
    lon2 = np.radians(np.asarray(lon2, dtype=float))
    dlat = lat2[:, None] - lat1[None, :]
    dlon = lon2[:, None] - lon1[None, :]
    a    = (np.sin(dlat/2)**2
            + np.cos(lat1[None, :]) * np.cos(lat2[:, None]) * np.sin(dlon/2)**2)
    return 2 * R_EARTH * np.arcsin(np.sqrt(a))

In [ ]:
# ── Prepare datasets ───────────────────────────────────────────────────────────
desert_zips = df[df['is_desert']]['zipcode'].tolist()

# Existing facilities in desert ZIPs (drop zero-capacity — same as Task 1)
desert_facilities = child_care_regulated_nyc[
    child_care_regulated_nyc['zipcode'].isin(desert_zips) &
    (child_care_regulated_nyc['total_capacity'] > 0)
].copy().reset_index(drop=True)

# Potential locations — assign a unique integer loc_id = row index
pot_locs = potential_locations_nyc.copy().reset_index(drop=True)
pot_locs['loc_id'] = pot_locs.index

print(f"Desert ZIPs                : {len(desert_zips)}")
print(f"Existing facilities        : {len(desert_facilities)}")
print(f"Total potential locations  : {len(pot_locs)}")

Desert ZIPs                : 179
Existing facilities        : 7739
Total potential locations  : 31100


In [ ]:
# ── Compute infeasible locations and forbidden pairs ───────────────────────────
infeasible_locs = set()  # loc_ids too close to an existing facility → cannot build here
forbidden_pairs = set()  # (loc_id_i, loc_id_j) → cannot build at BOTH locations

for z in desert_zips:
    ex = desert_facilities[desert_facilities['zipcode'] == z]
    pl = pot_locs[pot_locs['zipcode'] == z]

    if len(pl) == 0:
        continue

    pl_ids  = pl['loc_id'].values
    pl_lats = pl['latitude'].values
    pl_lons = pl['longitude'].values

    # ── 1. Existing facility vs Potential location ─────────────────────────────
    # If a potential location is within MIN_DIST of ANY existing facility
    # in the same ZIP → it is infeasible (we cannot build there)
    if len(ex) > 0:
        dist_ep = haversine(
            ex['latitude'].values, ex['longitude'].values,
            pl_lats, pl_lons
        )  # shape: (len(pl), len(ex))
        too_close = np.any(dist_ep < MIN_DIST, axis=1)  # True for each bad potential loc
        for loc_id, bad in zip(pl_ids, too_close):
            if bad:
                infeasible_locs.add(int(loc_id))

    # ── 2. Potential location vs Potential location ────────────────────────────
    # If two potential locations are within MIN_DIST of each other
    # → at most one can be chosen (forbidden pair constraint in the model)
    if len(pl) >= 2:
        dist_pp = haversine(pl_lats, pl_lons, pl_lats, pl_lons)
        # Upper triangle only (i < j) to avoid duplicates
        rows, cols = np.where(
            (dist_pp < MIN_DIST) &
            np.triu(np.ones_like(dist_pp, dtype=bool), k=1)
        )
        for i, j in zip(rows, cols):
            forbidden_pairs.add((int(pl_ids[i]), int(pl_ids[j])))

# Remove forbidden pairs that involve infeasible locations (already ruled out)
forbidden_pairs = {
    (i, j) for i, j in forbidden_pairs
    if i not in infeasible_locs and j not in infeasible_locs
}

# Final feasible potential locations
feasible_locs = pot_locs[~pot_locs['loc_id'].isin(infeasible_locs)].copy()

print(f"Total potential locations                          : {len(pot_locs)}")
print(f"Infeasible (too close to existing facility)       : {len(infeasible_locs)}")
print(f"Feasible potential locations remaining             : {len(feasible_locs)}")
print(f"Forbidden pairs among feasible potential locations : {len(forbidden_pairs)}")

Total potential locations                          : 31100
Infeasible (too close to existing facility)       : 3135
Feasible potential locations remaining             : 27965
Forbidden pairs among feasible potential locations : 87011


### 3.2 Decision Variables

#### Primary variables

| Variable | Type | Description |
|---|---|---|
| $x_{0\text{-}5,\, f} \geq 0$ | Integer | New 0–5 yr slots added to existing facility $f$ |
| $x_{5\text{-}12,\, f} \geq 0$ | Integer | New 5–12 yr slots added to existing facility $f$ |
| $b_{l,s} \in \{0,1\}$ | Binary | 1 if a new facility of size $s \in \{S, M, L\}$ is built at candidate location $l$ |
| $n_{0\text{-}5,\, z} \geq 0$ | Integer | Total 0–5 yr slots contributed by new builds in desert ZIP $z$ |
| $n_{5\text{-}12,\, z} \geq 0$ | Integer | Total 5–12 yr slots contributed by new builds in desert ZIP $z$ |

**New facility slot capacities:**

| Size | Total slots | Under-5 slots | Build cost |
|---|---|---|---|
| Small (S) | 100 | 50 | \$65,000 |
| Medium (M) | 200 | 100 | \$95,000 |
| Large (L) | 400 | 200 | \$115,000 |

#### Auxiliary piecewise variables

The expansion cost function is **non-continuous**: the *entire* expansion $x_f$ is priced at the rate of whichever tier it falls into — there is no incremental pricing. Cost jumps occur at each tier boundary.

To model this, we introduce binary tier selectors and auxiliary variables that capture the total expansion under each tier:

| Variable | Type | Description |
|---|---|---|
| $t_{1,f},\, t_{2,f},\, t_{3,f} \in \{0,1\}$ | Binary | Tier selector — which pricing bracket applies |
| $\delta_{1,f} \in [0,\; 0.10\,n_f]$ | Continuous | Total expansion when tier 1 is active |
| $\delta_{2,f} \in [0.10\,n_f,\; 0.15\,n_f]$ | Continuous | Total expansion when tier 2 is active |
| $\delta_{3,f} \in [0.15\,n_f,\; 0.20\,n_f]$ | Continuous | Total expansion when tier 3 is active |

At most one tier selector is active ($t_{1,f} + t_{2,f} + t_{3,f} \leq 1$), and:
$$x_{0\text{-}5,\,f} + x_{5\text{-}12,\,f} = \delta_{1,f} + \delta_{2,f} + \delta_{3,f}$$

Since exactly one $\delta_{k,f}$ is nonzero at a time, $\delta_{k,f} = x_f$ whenever tier $k$ is active. The cost is then $\text{rate}_k \times \delta_{k,f}$, which equals $\text{rate}_k \times x_f$ — correctly capturing the jump at each tier boundary.

In [ ]:
import gurobipy as gp
from gurobipy import GRB

# ── Cost / capacity parameters (same as Task 1) ───────────────────────────────
COST_S = 65_000;  CAP_S = 100;  UNDER5_CAP_S = 50
COST_M = 95_000;  CAP_M = 200;  UNDER5_CAP_M = 100
COST_L = 115_000; CAP_L = 400;  UNDER5_CAP_L = 200
EQUIP_COST = 100

# ── Index sets ─────────────────────────────────────────────────────────────────
cap_lookup  = desert_facilities.set_index('facility_id')['total_capacity'].to_dict()
fac_by_zip  = desert_facilities.groupby('zipcode')['facility_id'].apply(list).to_dict()
desert_df   = df[df['is_desert']].set_index('zipcode')

# Feasible locations per ZIP
feasible_locs['slots_0_5'] = (
    feasible_locs.get('infant_capacity', 0) +
    feasible_locs.get('toddler_capacity', 0) +
    feasible_locs.get('preschool_capacity', 0)
)
locs_by_zip = feasible_locs.groupby('zipcode')['loc_id'].apply(list).to_dict()

# ── Gurobi model ───────────────────────────────────────────────────────────────
m = gp.Model("ChildCareDesert_Task2")
m.setParam('OutputFlag', 1)

# ── Variable 1a: x_05[f] — 0-5 expansion slots at existing facility f ─────────
x_05  = m.addVars(list(cap_lookup.keys()), vtype=GRB.INTEGER, lb=0, name="x_05")

# ── Variable 1b: x_512[f] — 5-12 expansion slots at existing facility f ───────
x_512 = m.addVars(list(cap_lookup.keys()), vtype=GRB.INTEGER, lb=0, name="x_512")

# ── Variable 2: b[loc_id, size] — binary: build a facility of given size
#    at feasible location loc_id ─────────────────────────────────────────────────
all_loc_ids = feasible_locs['loc_id'].tolist()
b = {}
for size in ['S', 'M', 'L']:
    b[size] = m.addVars(all_loc_ids, vtype=GRB.BINARY, name=f"b_{size}")

# ── Variable 3a: n_05[z] — 0-5 slots from new builds in desert ZIP z ──────────
n_05  = m.addVars(desert_zips, vtype=GRB.INTEGER, lb=0, name="n_05")

# ── Variable 3b: n_512[z] — 5-12 slots from new builds in desert ZIP z ────────
n_512 = m.addVars(desert_zips, vtype=GRB.INTEGER, lb=0, name="n_512")

m.update()
print("Decision variables defined:")
print(f"  x_05     : {len(x_05):>6}  (0-5  expansion slots per existing facility)")
print(f"  x_512    : {len(x_512):>6}  (5-12 expansion slots per existing facility)")
print(f"  b_S      : {len(b['S']):>6}  (binary: build small  facility at location)")
print(f"  b_M      : {len(b['M']):>6}  (binary: build medium facility at location)")
print(f"  b_L      : {len(b['L']):>6}  (binary: build large  facility at location)")
print(f"  n_05     : {len(n_05):>6}  (0-5  slots from new builds per desert ZIP)")
print(f"  n_512    : {len(n_512):>6}  (5-12 slots from new builds per desert ZIP)")
print(f"\nTotal variables: {m.NumVars}")

Set parameter Username
Set parameter LicenseID to value 2773933
Academic license - for non-commercial use only - expires 2027-02-02
Set parameter OutputFlag to value 1
Decision variables defined:
  x_05     :   7739  (0-5  expansion slots per existing facility)
  x_512    :   7739  (5-12 expansion slots per existing facility)
  b_S      :  27965  (binary: build small  facility at location)
  b_M      :  27965  (binary: build medium facility at location)
  b_L      :  27965  (binary: build large  facility at location)
  n_05     :    179  (0-5  slots from new builds per desert ZIP)
  n_512    :    179  (5-12 slots from new builds per desert ZIP)

Total variables: 99731


### 3.3 Objective Function

Minimise total cost across three components:

$$\min \quad C_{\text{exp}} + C_{\text{build}} + C_{\text{equip}}$$

---

**Expansion cost** $C_{\text{exp}}$ — non-continuous piecewise cost. The *entire* expansion $x_f$ is priced at the rate of whichever tier it falls into (cost jumps at each boundary):

| Tier | Expansion range | Cost per slot (applied to all slots) |
|---|---|---|
| Tier 1 | $0 \leq x_f \leq 0.10\,n_f$ | $\dfrac{20{,}000 + 200\,n_f}{n_f}$ |
| Tier 2 | $0.10\,n_f < x_f \leq 0.15\,n_f$ | $\dfrac{20{,}000 + 400\,n_f}{n_f}$ |
| Tier 3 | $0.15\,n_f < x_f \leq 0.20\,n_f$ | $\dfrac{20{,}000 + 1{,}000\,n_f}{n_f}$ |

Since exactly one $\delta_{k,f} = x_f$ (the others are zero), the cost linearises as:

$$C_{\text{exp}} = \sum_{f} \left[ \frac{20{,}000 + 200\,n_f}{n_f}\,\delta_{1,f} \;+\; \frac{20{,}000 + 400\,n_f}{n_f}\,\delta_{2,f} \;+\; \frac{20{,}000 + 1{,}000\,n_f}{n_f}\,\delta_{3,f} \right]$$

---

**Construction cost** $C_{\text{build}}$ — fixed cost per new facility built:

$$C_{\text{build}} = \sum_{l} \left( 65{,}000\,b_{l,S} \;+\; 95{,}000\,b_{l,M} \;+\; 115{,}000\,b_{l,L} \right)$$

---

**Equipment surcharge** $C_{\text{equip}}$ — \$100 per 0–5 slot added (expansion or new build), reflecting the higher cost of infant/toddler equipment:

$$C_{\text{equip}} = 100 \cdot \sum_{z \in \mathcal{Z}} \left( \sum_{f \in z} x_{0\text{-}5,\,f} \;+\; n_{0\text{-}5,\,z} \right)$$

In [ ]:
# ── Piecewise expansion cost — non-continuous (jumps between tiers) ────────────
#
# The ENTIRE expansion x_f is priced at the rate of whichever tier it falls in:
#   Tier 1: x_f ∈ [0,       0.10·n_f]  → cost = rate1 × x_f
#   Tier 2: x_f ∈ (0.10·n_f, 0.15·n_f] → cost = rate2 × x_f  (jump at 10%)
#   Tier 3: x_f ∈ (0.15·n_f, 0.20·n_f] → cost = rate3 × x_f  (jump at 15%)
#
# Binary tier selectors t1/t2/t3 choose which tier applies.
# Auxiliary variables δ1/δ2/δ3 each equal x_f when their tier is active, else 0.
# Big-M bounds enforce both the upper limit and the lower limit (the jump) for
# tiers 2 and 3, ensuring x_f is genuinely within the selected tier's range.

t1 = m.addVars(list(cap_lookup.keys()), vtype=GRB.BINARY,     name="t1")
t2 = m.addVars(list(cap_lookup.keys()), vtype=GRB.BINARY,     name="t2")
t3 = m.addVars(list(cap_lookup.keys()), vtype=GRB.BINARY,     name="t3")
d1 = m.addVars(list(cap_lookup.keys()), vtype=GRB.CONTINUOUS, lb=0, name="delta1")
d2 = m.addVars(list(cap_lookup.keys()), vtype=GRB.CONTINUOUS, lb=0, name="delta2")
d3 = m.addVars(list(cap_lookup.keys()), vtype=GRB.CONTINUOUS, lb=0, name="delta3")

for f, n_f in cap_lookup.items():
    b1 = 0.10 * n_f   # upper bound of tier 1 / lower bound of tier 2
    b2 = 0.15 * n_f   # upper bound of tier 2 / lower bound of tier 3
    b3 = 0.20 * n_f   # hard cap (upper bound of tier 3)

    x_f = x_05[f] + x_512[f]

    # C8: hard expansion cap — no facility may expand beyond 20% of capacity
    m.addConstr(x_f <= b3,                             name=f"cap_{f}")

    # C7: at most one tier active; if none, x_f = 0 (no expansion)
    m.addConstr(t1[f] + t2[f] + t3[f] <= 1,           name=f"one_tier_{f}")

    # x_f equals whichever δ_k is active (the other two are forced to zero)
    m.addConstr(x_f == d1[f] + d2[f] + d3[f],         name=f"slot_split_{f}")

    # Tier 1: δ1 = x_f ∈ [0, b1]  when t1=1
    m.addConstr(d1[f] <= b1 * t1[f],                  name=f"d1_ub_{f}")

    # Tier 2: δ2 = x_f ∈ [b1, b2] when t2=1  — lower bound captures the jump
    m.addConstr(d2[f] >= b1 * t2[f],                  name=f"d2_lb_{f}")
    m.addConstr(d2[f] <= b2 * t2[f],                  name=f"d2_ub_{f}")

    # Tier 3: δ3 = x_f ∈ [b2, b3] when t3=1  — lower bound captures the jump
    m.addConstr(d3[f] >= b2 * t3[f],                  name=f"d3_lb_{f}")
    m.addConstr(d3[f] <= b3 * t3[f],                  name=f"d3_ub_{f}")

# ── Expansion cost ─────────────────────────────────────────────────────────────
# cost_f = rate_k × δ_k[f]  (equals rate_k × x_f since only one δ_k is nonzero)
C_exp = gp.quicksum(
    (20_000 + 200  * cap_lookup[f]) / cap_lookup[f] * d1[f]
  + (20_000 + 400  * cap_lookup[f]) / cap_lookup[f] * d2[f]
  + (20_000 + 1000 * cap_lookup[f]) / cap_lookup[f] * d3[f]
    for f in cap_lookup
)

# ── Construction cost ──────────────────────────────────────────────────────────
C_build = gp.quicksum(
    COST_S * b['S'][l] + COST_M * b['M'][l] + COST_L * b['L'][l]
    for l in all_loc_ids
)

# ── Equipment surcharge ($100 per 0-5 slot added) ─────────────────────────────
C_equip = EQUIP_COST * gp.quicksum(
    gp.quicksum(x_05[f] for f in fac_by_zip.get(z, []))
    + n_05[z]
    for z in desert_zips
)

m.setObjective(C_exp + C_build + C_equip, GRB.MINIMIZE)
m.update()
print("Objective set: minimize C_exp + C_build + C_equip")
print(f"  Binary tier selectors  (t1, t2, t3): {len(t1)} each")
print(f"  Expansion aux vars     (δ1, δ2, δ3): {len(d1)} each")

Objective set: minimize C_exp + C_build + C_equip
  Binary tier selectors  (t1, t2, t3): 7739 each
  Expansion aux vars     (δ1, δ2, δ3): 7739 each


### 3.4 Constraints

Let $\mathcal{Z}$ = set of 179 desert ZIP codes, $\mathcal{F}$ = set of forbidden location pairs, $n_f$ = current capacity of facility $f$.

---

**C1 — Total slot coverage** *(per desert ZIP)*

Each desert ZIP must receive enough new slots to close its total capacity deficit $D_z^{\text{total}}$:

$$\sum_{f \in z}\!\left(x_{0\text{-}5,f} + x_{5\text{-}12,f}\right) + \sum_{l \in z}\!\left(100\,b_{l,S} + 200\,b_{l,M} + 400\,b_{l,L}\right) \;\geq\; D_z^{\text{total}} \qquad \forall\,z \in \mathcal{Z}$$

where $D_z^{\text{total}} = \max\!\left(0,\; \tau_z \cdot P_z^{2\text{wk-}12} - \text{existing slots}_z\right)$, with $\tau_z = 1/2$ for High-demand ZIPs and $1/3$ for Normal.

---

**C2 — Under-5 slot coverage** *(per desert ZIP)*

Each desert ZIP must close its under-5 slot deficit $D_z^{0\text{-}5}$ (target: $2/3$ of children aged 0–5):

$$\sum_{f \in z} x_{0\text{-}5,\,f} \;+\; n_{0\text{-}5,\,z} \;\geq\; D_z^{0\text{-}5} \qquad \forall\,z \in \mathcal{Z}$$

---

**C3 — New build slot accounting** *(per desert ZIP)*

The slots assigned to new builds cannot exceed the total capacity of the facilities actually built in that ZIP:

$$n_{0\text{-}5,\,z} + n_{5\text{-}12,\,z} \;\leq\; \sum_{l \in z}\!\left(100\,b_{l,S} + 200\,b_{l,M} + 400\,b_{l,L}\right) \qquad \forall\,z \in \mathcal{Z}$$

---

**C4 — Under-5 capacity cap on new builds** *(per desert ZIP)*

$n_{0\text{-}5,z}$ cannot exceed the under-5 capacity of the facilities actually built:

$$n_{0\text{-}5,\,z} \;\leq\; \sum_{l \in z}\!\left(50\,b_{l,S} + 100\,b_{l,M} + 200\,b_{l,L}\right) \qquad \forall\,z \in \mathcal{Z}$$

---

**C5 — At most one size per location**

Each candidate location can host at most one facility:

$$b_{l,S} + b_{l,M} + b_{l,L} \;\leq\; 1 \qquad \forall\,l$$

---

**C6 — Minimum distance between new facilities**

For every forbidden pair $(i,j) \in \mathcal{F}$ (two candidate sites within 0.06 miles), at most one can be built:

$$\sum_{s \in \{S,M,L\}} b_{i,s} \;+\; \sum_{s \in \{S,M,L\}} b_{j,s} \;\leq\; 1 \qquad \forall\,(i,j) \in \mathcal{F}$$

---

**C7 — Piecewise tier constraints** *(per facility $f$, with $b_1 = 0.1n_f$, $b_2 = 0.15n_f$, $b_3 = 0.2n_f$)*

- **At most one tier active:** $\;t_{1,f} + t_{2,f} + t_{3,f} \leq 1$
- **Expansion decomposition:** $\;x_{0\text{-}5,\,f} + x_{5\text{-}12,\,f} = \delta_{1,f} + \delta_{2,f} + \delta_{3,f}$
- **Tier 1 bounds** (upper only, since expansion can start from 0): $\;0 \leq \delta_{1,f} \leq b_1\,t_{1,f}$
- **Tier 2 bounds** (both upper and lower, capturing the jump at $b_1$): $\;b_1\,t_{2,f} \leq \delta_{2,f} \leq b_2\,t_{2,f}$
- **Tier 3 bounds** (both upper and lower, capturing the jump at $b_2$): $\;b_2\,t_{3,f} \leq \delta_{3,f} \leq b_3\,t_{3,f}$

The lower bounds on $\delta_{2,f}$ and $\delta_{3,f}$ are what enforce the non-continuity: if tier 2 is selected, the entire expansion must be at least $b_1 = 0.1\,n_f$, and the full amount is priced at the tier 2 rate.

---

**C8 — Hard expansion cap** *(per facility $f$)*

No facility may expand beyond 20% of its current capacity:

$$x_{0\text{-}5,\,f} + x_{5\text{-}12,\,f} \;\leq\; 0.20\,n_f \qquad \forall\,f$$

*(This is implied by the tier bounds in C7, but stated explicitly for clarity.)*

In [ ]:
# ── Constraint 1: Total slot coverage per desert ZIP ──────────────────────────
for z in desert_zips:
    facs      = fac_by_zip.get(z, [])
    locs      = locs_by_zip.get(z, [])
    total_def = int(desert_df.loc[z, 'total_slot_deficit'])
    m.addConstr(
        gp.quicksum(x_05[f] + x_512[f] for f in facs)
        + gp.quicksum(
            CAP_S * b['S'][l] + CAP_M * b['M'][l] + CAP_L * b['L'][l]
            for l in locs)
        >= total_def,
        name=f"total_cov_{z}"
    )

# ── Constraint 2: Under-5 slot coverage per desert ZIP ────────────────────────
for z in desert_zips:
    facs       = fac_by_zip.get(z, [])
    under5_def = int(desert_df.loc[z, 'under5_slot_deficit'])
    m.addConstr(
        gp.quicksum(x_05[f] for f in facs) + n_05[z]
        >= under5_def,
        name=f"under5_cov_{z}"
    )

# ── Constraint 3: New build slot accounting ────────────────────────────────────
for z in desert_zips:
    locs = locs_by_zip.get(z, [])
    m.addConstr(
        n_05[z] + n_512[z]
        <= gp.quicksum(
            CAP_S * b['S'][l] + CAP_M * b['M'][l] + CAP_L * b['L'][l]
            for l in locs),
        name=f"build_cap_{z}"
    )

# ── Constraint 4: New build under-5 cap ───────────────────────────────────────
for z in desert_zips:
    locs = locs_by_zip.get(z, [])
    m.addConstr(
        n_05[z]
        <= gp.quicksum(
            UNDER5_CAP_S * b['S'][l] + UNDER5_CAP_M * b['M'][l] + UNDER5_CAP_L * b['L'][l]
            for l in locs),
        name=f"under5_cap_{z}"
    )

# ── Constraint 5: At most one facility size per location ──────────────────────
for l in all_loc_ids:
    m.addConstr(
        b['S'][l] + b['M'][l] + b['L'][l] <= 1,
        name=f"one_size_{l}"
    )

# ── Constraint 6: Distance — forbidden pairs of potential locations ────────────
for (i, j) in forbidden_pairs:
    m.addConstr(
        gp.quicksum(b[s][i] for s in ['S','M','L'])
        + gp.quicksum(b[s][j] for s in ['S','M','L'])
        <= 1,
        name=f"dist_{i}_{j}"
    )

m.update()
print("Constraints added:")
print(f"  1. Total slot coverage      : {len(desert_zips)}")
print(f"  2. Under-5 coverage         : {len(desert_zips)}")
print(f"  3. New build slot cap       : {len(desert_zips)}")
print(f"  4. New build under-5 cap    : {len(desert_zips)}")
print(f"  5. One size per location    : {len(all_loc_ids)}")
print(f"  6. Distance forbidden pairs : {len(forbidden_pairs)}")
print(f"\nTotal constraints: {m.NumConstrs}")

Constraints added:
  1. Total slot coverage      : 179
  2. Under-5 coverage         : 179
  3. New build slot cap       : 179
  4. New build under-5 cap    : 179
  5. One size per location    : 27965
  6. Distance forbidden pairs : 87011

Total constraints: 177604


In [ ]:
m.setParam('MIPGap', 0.0001)   # stop when gap ≤ 0.01% (already achieved)
m.setParam('TimeLimit', 120)    # hard stop at 120 seconds regardless


Set parameter MIPGap to value 0.0001
Set parameter TimeLimit to value 120


In [ ]:
m.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Core(TM) i5-7200U CPU @ 2.50GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads

Non-default parameters:
TimeLimit  120

Optimize a model with 177604 rows, 146165 columns and 1035459 nonzeros (Min)
Model fingerprint: 0x99b271c4
Model has 115030 linear objective coefficients
Variable types: 23217 continuous, 122948 integer (107112 binary)
Coefficient statistics:
  Matrix range     [3e-01, 4e+02]
  Objective range  [1e+02, 1e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [6e-01, 1e+04]

Presolved: 22060 rows, 32767 columns, 185493 nonzeros

Continuing optimization...

 172659  1535 2.3135e+08  721    1 2.3138e+08 2.3135e+08  0.01%   3.9  240s
 175815  3539 2.3135e+08  430    1 2.3138e+08 2.3135e+08  0.01%   3.8  245s
 177527  2968 2.3135e+08 1280    1 2.3138e+08 2.3135e+08  0.01%   3.8  250s
 179575  2285 2.3135e+08

In [1]:
if m.Status in (GRB.OPTIMAL, GRB.TIME_LIMIT) and m.SolCount > 0:
    total_cost  = m.ObjVal
    C_exp_val   = sum(
        (20_000 + 200  * cap_lookup[f]) / cap_lookup[f] * d1[f].X
      + (20_000 + 400  * cap_lookup[f]) / cap_lookup[f] * d2[f].X
      + (20_000 + 1000 * cap_lookup[f]) / cap_lookup[f] * d3[f].X
        for f in cap_lookup
    )
    C_build_val = sum(
        COST_S * b['S'][l].X + COST_M * b['M'][l].X + COST_L * b['L'][l].X
        for l in all_loc_ids
    )
    C_equip_val = EQUIP_COST * sum(
        sum(x_05[f].X for f in fac_by_zip.get(z, [])) + n_05[z].X
        for z in desert_zips
    )

    total_exp_05  = sum(x_05[f].X  for f in cap_lookup)
    total_exp_512 = sum(x_512[f].X for f in cap_lookup)
    new_S = sum(b['S'][l].X for l in all_loc_ids)
    new_M = sum(b['M'][l].X for l in all_loc_ids)
    new_L = sum(b['L'][l].X for l in all_loc_ids)

    status_label = "OPTIMAL" if m.Status == GRB.OPTIMAL else f"BEST FEASIBLE (gap {m.MIPGap*100:.4f}%)"

    print("=" * 60)
    print(f"              {status_label}")
    print("=" * 60)
    print(f"\n  MINIMUM FUNDING REQUIRED (Realistic Scenario)")
    print(f"  ${total_cost:,.2f}")
    print(f"\n  Cost breakdown:")
    print(f"  ├─ Expansion Cost        : ${C_exp_val:>15,.2f}")
    print(f"  ├─ Construction Cost     : ${C_build_val:>15,.2f}")
    print(f"  └─ Equipment Surcharge   : ${C_equip_val:>15,.2f}")
    print(f"\n  Solution details:")
    print(f"  Expansion slots (0-5)   : {total_exp_05:>10,.0f}")
    print(f"  Expansion slots (5-12)  : {total_exp_512:>10,.0f}")
    print(f"  New facilities built    : {new_S+new_M+new_L:>10,.0f}")
    print(f"    ├─ Small  (100 slots) : {new_S:>10,.0f}")
    print(f"    ├─ Medium (200 slots) : {new_M:>10,.0f}")
    print(f"    └─ Large  (400 slots) : {new_L:>10,.0f}")
    print(f"\n{'=' * 60}")
    print(f"  The minimum amount of funding needed to eliminate all")
    print(f"  child-care deserts in NYC under the realistic scenario")
    print(f"  (piecewise expansion costs + minimum distance constraints)")
    print(f"  is: ${total_cost:,.2f}")
    print(f"{'=' * 60}")

elif m.Status == GRB.INFEASIBLE:
    print("Model is INFEASIBLE")
    m.computeIIS()
    m.write("infeasible.ilp")
else:
    print(f"Solver status: {m.Status}")

NameError: name 'm' is not defined